In [156]:
import pandas as pd
import utils
import plotly.graph_objects as go
import instrument

In [157]:
pair = "USD_JPY"
granularity = "H1"
ma_list = [16,64]

i_pair = instrument.Instrument.get_instrument_by_name(pair)

In [158]:
df = pd.read_pickle(utils.get_his_data_filename(pair, granularity))
non_cols = ['time', 'volume']
mod_cols = [x for x in df.columns if x not in non_cols]
df[mod_cols] = df[mod_cols].apply(pd.to_numeric)

In [159]:

df_ma = df[['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()
for ma in ma_list:  
    df_ma[f'MA_{ma}'] = df_ma.mid_c.rolling(window=ma).mean()
df_ma.dropna(inplace=True) #Parameter actually changes the data to remove data with NaN values. This is important because the first 7 rows will have NaN values
df_ma.reset_index(drop=True, inplace=True)

In [160]:
df_ma.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_16,MA_64
0,2026-03-03T14:00:00.000000000Z,157.804,157.954,157.717,157.717,157.492375,156.750781
1,2026-03-03T15:00:00.000000000Z,157.718,157.916,157.602,157.844,157.528250,156.781781
2,2026-03-03T16:00:00.000000000Z,157.844,157.928,157.612,157.699,157.541688,156.810047
3,2026-03-03T17:00:00.000000000Z,157.698,157.868,157.676,157.836,157.575687,156.838984
4,2026-03-03T18:00:00.000000000Z,157.836,157.846,157.504,157.653,157.600875,156.868750


In [161]:
def is_trade(row):
    if row.DIFF >= 0 and row.DIFF_PREV < 0:
        return True
    if row.DIFF <= 0 and row.DIFF_PREV > 0:
        return True
    return False


In [162]:
df_ma['DIFF'] = df_ma.MA_16 - df_ma.MA_64
df_ma['DIFF_PREV'] = df_ma.DIFF.shift(1)
df_ma['IS_TRADE'] = df_ma.apply(is_trade, axis=1)
df_trades = df_ma[df_ma.IS_TRADE==True].copy()

In [163]:
df_trades.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_16,MA_64,DIFF,DIFF_PREV,IS_TRADE
27,2026-03-04T17:00:00.000000000Z,156.944,157.039,156.882,156.977,157.304750,157.353281,-0.048531,0.001609,True
53,2026-03-05T19:00:00.000000000Z,157.733,157.849,157.694,157.814,157.432688,157.394250,0.038438,-0.004578,True
111,2026-03-10T04:00:00.000000000Z,157.890,157.932,157.764,157.778,157.955125,157.970031,-0.014906,0.034938,True
139,2026-03-11T08:00:00.000000000Z,158.224,158.447,158.199,158.404,158.113563,158.077344,0.036219,-0.006250,True
221,2026-03-16T18:00:00.000000000Z,159.272,159.272,158.912,158.965,159.251625,159.277859,-0.026234,0.011422,True


In [164]:
df_trades["DELTA"] = (df_trades.mid_c.diff() / i_pair.pipLocation).shift(-1)
df_trades["GAINS"] = df_trades["DELTA"] * df_trades["IS_TRADE"]

In [169]:
from dateutil.parser import *
df_trades["time"] = [parse(x) for x in df_trades.time] 
df_trades["DURATION"] = df_trades.time.diff().shift(-1)
df_trades["DURATION"] = [x.total_seconds() / 3600 for x in df_trades.DURATION]
df_trades.dropna(inplace=True)

In [170]:
df_trades.info()

<class 'pandas.DataFrame'>
Index: 77 entries, 27 to 3877
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype                  
---  ------     --------------  -----                  
 0   time       77 non-null     datetime64[us, tzutc()]
 1   mid_o      77 non-null     float64                
 2   mid_h      77 non-null     float64                
 3   mid_l      77 non-null     float64                
 4   mid_c      77 non-null     float64                
 5   MA_16      77 non-null     float64                
 6   MA_64      77 non-null     float64                
 7   DIFF       77 non-null     float64                
 8   DIFF_PREV  77 non-null     float64                
 9   IS_TRADE   77 non-null     bool                   
 10  DELTA      77 non-null     float64                
 11  GAINS      77 non-null     float64                
 12  DURATION   77 non-null     float64                
dtypes: bool(1), datetime64[us, tzutc()](1), float64(11)
memory usage:

In [ ]:
df_plot = df_ma.iloc[0:24].copy() #get the last 100 rows of the dataframe and make a copy of it

In [167]:
df_plot.shape

(550, 10)

In [168]:
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df_plot.time, open=df_plot.mid_o, high=df_plot.mid_h, low=df_plot.mid_l, close=df_plot.mid_c,
    line=dict(width=1), opacity=1,
    increasing_fillcolor='#24A06B',
    decreasing_fillcolor="#CC2E3C",
    increasing_line_color='#2EC886',  
    decreasing_line_color='#FF3A4C'
))

for ma in ma_list:  
    col = f"MA_{ma}"
    fig.add_trace(go.Scatter(x=df_plot.time, 
        y=df_plot[col],
        line=dict(width=2),
        line_shape='spline',
        name=col
        ))
#Will give us a line plot of the moving average on top of the candlestick chart


fig.update_layout(width=1100, height=500,
    margin=dict(l=10, r=10, t=10, b=10),
    font=dict(color="#e1e1e1"),
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e"
)

fig.update_xaxes(
    gridcolor="#1f292f",
    showgrid=True,fixedrange=True,rangeslider=dict(visible=False)
)

fig.update_yaxes(
    gridcolor="#1f292f",
    showgrid=True
)

fig.show()